# 01 - Composite plate PINN-PC (reduced KL)

This notebook trains a cleaned PINN-PC model using reduced KL modes (4 + 4) and compares against the FEM Monte Carlo reference.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import deepxde as dde
from scipy.interpolate import RegularGridInterpolator

from phd.plot import get_current_config as plt_cfg, book_config, KUL_CYCLE

book_config.set_as_current()
page_width = plt_cfg().page_width
mpl.rcParams["axes.prop_cycle"] = mpl.cycler(color=KUL_CYCLE)
mpl.rcParams["image.cmap"] = "viridis"

save_fig = True
if save_fig:
    mpl.rcParams["pgf.texsystem"] = "pdflatex"

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for parent in [current, *current.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("Could not find project root (missing pyproject.toml).")

PROJECT_ROOT = find_project_root(Path.cwd())
CHAPTER_DIR = PROJECT_ROOT / "chapters" / "V_UncertaintyPropagation" / "02_composite_plate"
IMAGE_DIR = CHAPTER_DIR / "images"
PGF_DIR = IMAGE_DIR / "pgf"
PNG_DIR = IMAGE_DIR / "png"
DATA_DIR = CHAPTER_DIR / "data"

def save_figure(fig, name: str, dpi: int = 300):
    if not save_fig:
        return
    fig.savefig(PGF_DIR / f"{name}.pgf", bbox_inches="tight")
    fig.savefig(PNG_DIR / f"{name}.png", bbox_inches="tight", dpi=dpi)

## Load FEM reference and KL representation

In [ ]:
SEED = 42
np.random.seed(SEED)
dde.config.set_random_seed(SEED)

ref_path = DATA_DIR / "mc_reference_plate.npz"
if not ref_path.exists():
    raise FileNotFoundError("Run 01_FEM.ipynb first to generate mc_reference_plate.npz")

ref = np.load(ref_path)
xy_ref = ref["xy"]
u_mean_ref = ref["u_mean"]
u_std_ref = ref["u_std"]
x_grid = ref["x_grid"]
y_grid = ref["y_grid"]
phi11 = ref["phi11"]
phi33 = ref["phi33"]
lam11 = ref["lam11"]
lam33 = ref["lam33"]

C11_MU = float(ref["c11_mu"])
C11_SIGMA = float(ref["c11_sigma"])
C33_MU = float(ref["c33_mu"])
C33_SIGMA = float(ref["c33_sigma"])

L_PLATE = float(ref["L"])
P_STRESS = float(ref["p_stress"])
M_FIELD = int(ref["M_field"])
M_TOTAL = 2 * M_FIELD

print("Loaded reference data from:", ref_path)
print("M_FIELD =", M_FIELD, "(total stochastic dimension =", M_TOTAL, ")")

## PINN-PC setup

In [ ]:
n_colloc = 80
x_line = np.linspace(0.0, L_PLATE, n_colloc)
y_line = np.linspace(0.0, L_PLATE, n_colloc)
Xc, Yc = np.meshgrid(x_line, y_line, indexing="ij")
X_domain = np.column_stack([Xc.ravel(), Yc.ravel()])

# Mode values and mode gradients at collocation points.
def mode_values_and_grads(phi: np.ndarray):
    n_modes = phi.shape[-1]
    vals = np.zeros((X_domain.shape[0], n_modes))
    dvals_dx = np.zeros((X_domain.shape[0], n_modes))
    dvals_dy = np.zeros((X_domain.shape[0], n_modes))

    for k in range(n_modes):
        mode = phi[:, :, k]
        dmode_dx = np.gradient(mode, x_grid, axis=0)
        dmode_dy = np.gradient(mode, y_grid, axis=1)
        interp_mode = RegularGridInterpolator((x_grid, y_grid), mode)
        interp_dx = RegularGridInterpolator((x_grid, y_grid), dmode_dx)
        interp_dy = RegularGridInterpolator((x_grid, y_grid), dmode_dy)
        vals[:, k] = interp_mode(X_domain)
        dvals_dx[:, k] = interp_dx(X_domain)
        dvals_dy[:, k] = interp_dy(X_domain)

    return vals, dvals_dx, dvals_dy

phi11_col, dphi11_dx_col, dphi11_dy_col = mode_values_and_grads(phi11)
phi33_col, dphi33_dx_col, dphi33_dy_col = mode_values_and_grads(phi33)

n_xi_samples = 10000
n_xi_batch = 512
xi_train = np.random.randn(n_xi_samples, M_TOTAL).astype(np.float32)

dtype = dde.backend.data_type_dict["float32"]
xi_train_t = dde.backend.as_tensor(xi_train, dtype=dtype)
phi11_t = dde.backend.as_tensor(phi11_col, dtype=dtype)
phi33_t = dde.backend.as_tensor(phi33_col, dtype=dtype)
dphi11_dx_t = dde.backend.as_tensor(dphi11_dx_col, dtype=dtype)
dphi11_dy_t = dde.backend.as_tensor(dphi11_dy_col, dtype=dtype)
dphi33_dx_t = dde.backend.as_tensor(dphi33_dx_col, dtype=dtype)
dphi33_dy_t = dde.backend.as_tensor(dphi33_dy_col, dtype=dtype)
sqrt_lam11_t = dde.backend.as_tensor(np.sqrt(lam11).astype(np.float32), dtype=dtype)
sqrt_lam33_t = dde.backend.as_tensor(np.sqrt(lam33).astype(np.float32), dtype=dtype)

idx_left = np.where(np.isclose(X_domain[:, 0], 0.0))[0]
idx_bottom = np.where(np.isclose(X_domain[:, 1], 0.0))[0]
idx_right = np.where(np.isclose(X_domain[:, 0], L_PLATE))[0]
idx_top = np.where(np.isclose(X_domain[:, 1], L_PLATE))[0]
idx_boundary = np.where(
    np.isclose(X_domain[:, 0], 0.0)
    | np.isclose(X_domain[:, 0], L_PLATE)
    | np.isclose(X_domain[:, 1], 0.0)
    | np.isclose(X_domain[:, 1], L_PLATE)
)[0]
idx_interior = np.setdiff1d(np.arange(X_domain.shape[0]), idx_boundary)

In [ ]:
def jac(y, x, comp, j):
    out = dde.grad.jacobian(y, x, i=comp, j=j)
    return out[0] if dde.backend.backend_name == "jax" else out

def hess(y, x, comp, i, j):
    out = dde.grad.hessian(y, x, component=comp, i=i, j=j)
    return out[0] if dde.backend.backend_name == "jax" else out

def split_output(y):
    ux_mean = y[:, 0:1]
    ux_coef = y[:, 1 : 1 + M_TOTAL]
    uy_mean = y[:, 1 + M_TOTAL : 2 + M_TOTAL]
    uy_coef = y[:, 2 + M_TOTAL :]
    return ux_mean, ux_coef, uy_mean, uy_coef

def field_from_kl_batch(xi_batch):
    xi11 = xi_batch[:, :M_FIELD]
    xi33 = xi_batch[:, M_FIELD:]

    g11 = phi11_t @ (sqrt_lam11_t * dde.backend.transpose(xi11))
    g33 = phi33_t @ (sqrt_lam33_t * dde.backend.transpose(xi33))

    C11 = dde.backend.exp(C11_MU + C11_SIGMA * g11)
    C33 = dde.backend.exp(C33_MU + C33_SIGMA * g33)

    dg11_dx = dphi11_dx_t @ (sqrt_lam11_t * dde.backend.transpose(xi11))
    dg11_dy = dphi11_dy_t @ (sqrt_lam11_t * dde.backend.transpose(xi11))
    dg33_dx = dphi33_dx_t @ (sqrt_lam33_t * dde.backend.transpose(xi33))
    dg33_dy = dphi33_dy_t @ (sqrt_lam33_t * dde.backend.transpose(xi33))

    dC11_dx = C11_SIGMA * dg11_dx * C11
    dC11_dy = C11_SIGMA * dg11_dy * C11
    dC33_dx = C33_SIGMA * dg33_dx * C33
    dC33_dy = C33_SIGMA * dg33_dy * C33

    return (
        dde.backend.transpose(C11),
        dde.backend.transpose(C33),
        dde.backend.transpose(dC11_dx),
        dde.backend.transpose(dC11_dy),
        dde.backend.transpose(dC33_dx),
        dde.backend.transpose(dC33_dy),
    )

def pde_plate_pc(x, y):
    indices = np.random.permutation(xi_train.shape[0])[:n_xi_batch]
    xi_batch = xi_train_t[indices]

    ux_mean, ux_coef, uy_mean, uy_coef = split_output(y)

    dux_dx_mean = jac(y, x, 0, 0)
    dux_dy_mean = jac(y, x, 0, 1)
    duy_dx_mean = jac(y, x, 1 + M_TOTAL, 0)
    duy_dy_mean = jac(y, x, 1 + M_TOTAL, 1)

    d2ux_dxdx_mean = hess(y, x, 0, 0, 0)
    d2ux_dxdy_mean = hess(y, x, 0, 0, 1)
    d2ux_dydy_mean = hess(y, x, 0, 1, 1)
    d2uy_dxdx_mean = hess(y, x, 1 + M_TOTAL, 0, 0)
    d2uy_dxdy_mean = hess(y, x, 1 + M_TOTAL, 0, 1)
    d2uy_dydy_mean = hess(y, x, 1 + M_TOTAL, 1, 1)

    dux_dx_coef = []
    dux_dy_coef = []
    duy_dx_coef = []
    duy_dy_coef = []

    d2ux_dxdx_coef = []
    d2ux_dxdy_coef = []
    d2ux_dydy_coef = []
    d2uy_dxdx_coef = []
    d2uy_dxdy_coef = []
    d2uy_dydy_coef = []

    for k in range(M_TOTAL):
        cu = 1 + k
        cv = 2 + M_TOTAL + k
        dux_dx_coef.append(jac(y, x, cu, 0))
        dux_dy_coef.append(jac(y, x, cu, 1))
        duy_dx_coef.append(jac(y, x, cv, 0))
        duy_dy_coef.append(jac(y, x, cv, 1))

        d2ux_dxdx_coef.append(hess(y, x, cu, 0, 0))
        d2ux_dxdy_coef.append(hess(y, x, cu, 0, 1))
        d2ux_dydy_coef.append(hess(y, x, cu, 1, 1))
        d2uy_dxdx_coef.append(hess(y, x, cv, 0, 0))
        d2uy_dxdy_coef.append(hess(y, x, cv, 0, 1))
        d2uy_dydy_coef.append(hess(y, x, cv, 1, 1))

    dux_dx_coef = dde.backend.stack(dux_dx_coef, axis=1).squeeze()
    dux_dy_coef = dde.backend.stack(dux_dy_coef, axis=1).squeeze()
    duy_dx_coef = dde.backend.stack(duy_dx_coef, axis=1).squeeze()
    duy_dy_coef = dde.backend.stack(duy_dy_coef, axis=1).squeeze()

    d2ux_dxdx_coef = dde.backend.stack(d2ux_dxdx_coef, axis=1).squeeze()
    d2ux_dxdy_coef = dde.backend.stack(d2ux_dxdy_coef, axis=1).squeeze()
    d2ux_dydy_coef = dde.backend.stack(d2ux_dydy_coef, axis=1).squeeze()
    d2uy_dxdx_coef = dde.backend.stack(d2uy_dxdx_coef, axis=1).squeeze()
    d2uy_dxdy_coef = dde.backend.stack(d2uy_dxdy_coef, axis=1).squeeze()
    d2uy_dydy_coef = dde.backend.stack(d2uy_dydy_coef, axis=1).squeeze()

    dux_dx = dux_dx_mean + xi_batch @ dde.backend.transpose(dux_dx_coef)
    dux_dy = dux_dy_mean + xi_batch @ dde.backend.transpose(dux_dy_coef)
    duy_dx = duy_dx_mean + xi_batch @ dde.backend.transpose(duy_dx_coef)
    duy_dy = duy_dy_mean + xi_batch @ dde.backend.transpose(duy_dy_coef)

    d2ux_dxdx = d2ux_dxdx_mean + xi_batch @ dde.backend.transpose(d2ux_dxdx_coef)
    d2ux_dxdy = d2ux_dxdy_mean + xi_batch @ dde.backend.transpose(d2ux_dxdy_coef)
    d2ux_dydy = d2ux_dydy_mean + xi_batch @ dde.backend.transpose(d2ux_dydy_coef)
    d2uy_dxdx = d2uy_dxdx_mean + xi_batch @ dde.backend.transpose(d2uy_dxdx_coef)
    d2uy_dxdy = d2uy_dxdy_mean + xi_batch @ dde.backend.transpose(d2uy_dxdy_coef)
    d2uy_dydy = d2uy_dydy_mean + xi_batch @ dde.backend.transpose(d2uy_dydy_coef)

    Exx = dux_dx
    Eyy = duy_dy
    gamma_xy = dux_dy + duy_dx

    dExx_dx = d2ux_dxdx
    dExx_dy = d2ux_dxdy
    dEyy_dx = d2uy_dxdy
    dEyy_dy = d2uy_dydy

    dgamma_dx = d2ux_dxdy + d2uy_dxdx
    dgamma_dy = d2ux_dydy + d2uy_dxdy

    C11, C33, dC11_dx, dC11_dy, dC33_dx, dC33_dy = field_from_kl_batch(xi_batch)

    Sxx = C11 * (Exx + Eyy) - 2.0 * C33 * Eyy
    Syy = C11 * (Exx + Eyy) - 2.0 * C33 * Exx
    Sxy = C33 * gamma_xy

    dSxx_dx = dC11_dx * (Exx + Eyy) + C11 * (dExx_dx + dEyy_dx) - 2.0 * dC33_dx * Eyy - 2.0 * C33 * dEyy_dx
    dSyy_dy = dC11_dy * (Exx + Eyy) + C11 * (dExx_dy + dEyy_dy) - 2.0 * dC33_dy * Exx - 2.0 * C33 * dExx_dy
    dSxy_dx = dC33_dx * gamma_xy + C33 * dgamma_dx
    dSxy_dy = dC33_dy * gamma_xy + C33 * dgamma_dy

    Mx = dSxx_dx + dSxy_dy
    My = dSxy_dx + dSyy_dy

    res_mx = Mx[:, idx_interior]
    res_my = My[:, idx_interior]
    bc_tx = Sxx[:, idx_right] - P_STRESS
    bc_ty = Syy[:, idx_top]
    bc_shear = Sxy[:, idx_boundary]

    return [res_mx, res_my, bc_tx, bc_ty, bc_shear]

def hard_bc_output(x, y):
    ux_block = y[:, : 1 + M_TOTAL] * x[:, 0:1] / L_PLATE
    uy_block = y[:, 1 + M_TOTAL :] * x[:, 1:2] / L_PLATE
    return dde.backend.concat([ux_block, uy_block], axis=1)

geom = dde.geometry.Rectangle([0.0, 0.0], [L_PLATE, L_PLATE])
data = dde.data.PDE(
    geom,
    pde_plate_pc,
    bcs=[],
    num_domain=0,
    anchors=X_domain,
    num_test=X_domain.shape[0],
)

output_dim = 2 * (1 + M_TOTAL)
layers = [2, [64] * output_dim, [64] * output_dim, [64] * output_dim, output_dim]
net = dde.nn.PFNN(layers, "tanh", "Glorot uniform", regularization=["l2", 1e-6])
net.apply_output_transform(hard_bc_output)

model = dde.Model(data, net)
model.compile("adam", lr=1e-3, loss_weights=[1.0, 1.0, 5.0, 5.0, 2.0])

In [ ]:
n_iter = 50000
losshistory, train_state = model.train(iterations=n_iter, display_every=1000)

## Post-processing and comparison with FEM Monte Carlo

In [ ]:
# Evaluate model on a dense grid.
n_eval = 120
x_eval = np.linspace(0.0, L_PLATE, n_eval)
y_eval = np.linspace(0.0, L_PLATE, n_eval)
Xe, Ye = np.meshgrid(x_eval, y_eval, indexing="ij")
X_eval = np.column_stack([Xe.ravel(), Ye.ravel()])

Y_eval = model.predict(X_eval)

ux_mean_eval = Y_eval[:, 0:1]
ux_coef_eval = Y_eval[:, 1 : 1 + M_TOTAL]
uy_mean_eval = Y_eval[:, 1 + M_TOTAL : 2 + M_TOTAL]
uy_coef_eval = Y_eval[:, 2 + M_TOTAL :]

xi_plot = np.random.randn(20000, M_TOTAL)
ux_mc = ux_mean_eval + ux_coef_eval @ xi_plot.T
uy_mc = uy_mean_eval + uy_coef_eval @ xi_plot.T

ux_pred_mean = np.mean(ux_mc, axis=1).reshape(n_eval, n_eval)
uy_pred_mean = np.mean(uy_mc, axis=1).reshape(n_eval, n_eval)
ux_pred_std = np.std(ux_mc, axis=1).reshape(n_eval, n_eval)
uy_pred_std = np.std(uy_mc, axis=1).reshape(n_eval, n_eval)

# Interpolate FEM reference on the same evaluation grid.
x_ref_u = np.unique(xy_ref[:, 0])
y_ref_u = np.unique(xy_ref[:, 1])
nx_ref, ny_ref = len(x_ref_u), len(y_ref_u)
ux_mean_ref_grid = u_mean_ref[:, 0].reshape(nx_ref, ny_ref)
uy_mean_ref_grid = u_mean_ref[:, 1].reshape(nx_ref, ny_ref)
ux_std_ref_grid = u_std_ref[:, 0].reshape(nx_ref, ny_ref)
uy_std_ref_grid = u_std_ref[:, 1].reshape(nx_ref, ny_ref)

interp_ux_mean = RegularGridInterpolator((x_ref_u, y_ref_u), ux_mean_ref_grid)
interp_uy_mean = RegularGridInterpolator((x_ref_u, y_ref_u), uy_mean_ref_grid)
interp_ux_std = RegularGridInterpolator((x_ref_u, y_ref_u), ux_std_ref_grid)
interp_uy_std = RegularGridInterpolator((x_ref_u, y_ref_u), uy_std_ref_grid)

ux_ref_eval = interp_ux_mean(X_eval).reshape(n_eval, n_eval)
uy_ref_eval = interp_uy_mean(X_eval).reshape(n_eval, n_eval)
ux_std_ref_eval = interp_ux_std(X_eval).reshape(n_eval, n_eval)
uy_std_ref_eval = interp_uy_std(X_eval).reshape(n_eval, n_eval)

In [ ]:
fig_pc, axs = plt.subplots(2, 2, figsize=(0.8 * page_width, 0.65 * page_width), dpi=300, constrained_layout=True)
fields = [ux_pred_mean, uy_pred_mean, ux_pred_std, uy_pred_std]
titles = [r"$u_x$ mean (PINN-PC)", r"$u_y$ mean (PINN-PC)", r"$u_x$ std (PINN-PC)", r"$u_y$ std (PINN-PC)"]
for ax, field, title in zip(axs.flat, fields, titles):
    im = ax.pcolormesh(Xe, Ye, field, shading="auto")
    ax.set_title(title)
    ax.set_aspect("equal")
    fig_pc.colorbar(im, ax=ax)
save_figure(fig_pc, "PC_mean_std")
plt.show()

fig_cmp, axs_cmp = plt.subplots(2, 4, figsize=(1.2 * page_width, 0.65 * page_width), dpi=300, constrained_layout=True)
pred_fields = [ux_pred_mean, uy_pred_mean, ux_pred_std, uy_pred_std]
ref_fields = [ux_ref_eval, uy_ref_eval, ux_std_ref_eval, uy_std_ref_eval]
names = [r"$u_x$ mean", r"$u_y$ mean", r"$u_x$ std", r"$u_y$ std"]
for j in range(4):
    im0 = axs_cmp[0, j].pcolormesh(Xe, Ye, ref_fields[j], shading="auto")
    axs_cmp[0, j].set_title(names[j] + " (FEM)")
    axs_cmp[0, j].set_aspect("equal")
    fig_cmp.colorbar(im0, ax=axs_cmp[0, j])

    im1 = axs_cmp[1, j].pcolormesh(Xe, Ye, pred_fields[j], shading="auto")
    axs_cmp[1, j].set_title(names[j] + " (PINN-PC)")
    axs_cmp[1, j].set_aspect("equal")
    fig_cmp.colorbar(im1, ax=axs_cmp[1, j])
plt.show()